In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
BRONZE_PATH = "abfss://bronze@pravdatalake.dfs.core.windows.net"
SILVER_PATH = "abfss://silver@pravdatalake.dfs.core.windows.net"

In [0]:
titles_df = spark.read.format("delta")\
    .option("header", True)\
    .option("inferSchema", True)\
    .load(f"{BRONZE_PATH}/netflix_titles")

In [0]:
print("Titles:", titles_df.count())

In [0]:
titles_df.display()

In [0]:
silver_titles = titles_df.withColumn(
    "duration_minutes",
    expr("try_cast(duration_minutes AS INT)")
).withColumn(
    "duration_seasons",
    expr("try_cast(duration_seasons AS INT)")
)

In [0]:
silver_titles = (
    titles_df

    # Clean string columns
    .withColumn("show_id", trim(col("show_id")))
    .withColumn("type", trim(col("type")))
    .withColumn("title", trim(col("title")))
    .withColumn("rating", trim(col("rating")))
    .withColumn("description",trim(col("description")))

    # Convert empty strings to NULL
    .withColumn(
        "type",
        when(col("type") == "", None).otherwise(col("type"))
    )
    .withColumn(
        "title",
        when(col("title") == "", None).otherwise(col("title"))
    )
    .withColumn(
        "rating",
        when(col("rating") == "", None).otherwise(col("rating"))
    )
    .withColumn(
        "description",
        when(col("description") == "", None).otherwise(col("description"))
    )

    # Standardize type
    .withColumn(
        "type",
        when(col("type").isin("Movie", "TV Show"), col("type"))
         .otherwise(None)
    )


    # Convert date
    .withColumn(
        "date_added",
        to_date(trim(col("date_added")), "M/d/yyyy")
    )

    # Derived date attributes
    .withColumn("added_year", year("date_added"))
    .withColumn("added_month", month("date_added"))
    .withColumn("added_quarter", quarter("date_added"))

    # Remove completely invalid title records
    .filter(col("show_id").isNotNull())

    # Deduplicate
    .dropDuplicates(["show_id"])
)

####Validate

In [0]:
silver_titles.printSchema()

silver_titles.select(
    "show_id",
    "type",
    "title",
    "release_year",
    "rating",
    "duration_minutes",
    "duration_seasons",
    "date_added"
).display()

In [0]:
silver_titles = (
    silver_titles
    .withColumn(
        "duration_minutes",
        when(
            col("type") == "Movie",
            col("duration_minutes")
        )
    )
    .withColumn(
        "duration_seasons",
        when(
            col("type") == "TV Show",
            col("duration_seasons")
        )
    )
)

####Adiing Metadata

In [0]:
silver_titles = (
    silver_titles
    .withColumn("silver_ingestion_timestamp", current_timestamp())
    .withColumn("silver_source", lit("netflix_titles"))
)

In [0]:
silver_titles.display()

In [0]:
silver_titles.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{SILVER_PATH}/netflix_titles")